# 02/SageMaker 멀티모달 SFT (이미지→JSON): 멀티모달 추출

**요약**: `scripts/train_mm.py`를 SageMaker ModelTrainer로 실행합니다. gemma-4를 vision 포함 로드하고, vision tower는 얼린 채 language LoRA만 학습합니다.

**목적**: 이미지→텍스트 파인튜닝은 processor(이미지 전처리)+멀티모달 모델 클래스가 필요합니다. train_mm.py가 TRL SFTTrainer에 processor를 넘겨 이미지를 자동 처리하고, vision tower를 freeze해 안정적으로 학습합니다.

**배경**: 멀티모달 base에 무작정 all-linear LoRA를 붙이면 vision proj(ClippableLinear)에서 크래시합니다. language_model 한정 regex target으로 이를 피합니다(실측 검증).

> 실제 실행에는 AWS 자격증명과 비용이 필요합니다. 먼저 `DRY_RUN=1`로 파이프라인을 검증하세요.

In [ ]:
import os, sys
# 리포 루트를 path에 추가해 common/ 와 트랙 로컬 모듈을 import
REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, REPO)
sys.path.insert(0, os.getcwd())

In [ ]:
import importlib, boto3
from common import config, dlc, aws_utils; importlib.reload(config)
from sagemaker.core.helper.session_helper import Session
from sagemaker.train.model_trainer import ModelTrainer
from sagemaker.core.training.configs import SourceCode, Compute, StoppingCondition
sess = Session(boto3.Session(region_name=config.AWS_REGION))
%store -r role
if 'role' not in dir() or not role or ':role/' not in str(role):
    role = config.resolve_sagemaker_role(sess)
TRACK = config.TRACKS['mm_extraction']
print('role:', role, '| seed:', TRACK.seed_dataset, '| multimodal:', TRACK.multimodal)

## 학습 구성
이 트랙은 시드 이미지 데이터셋(cord-v2)을 컨테이너 안에서 직접 로드합니다(합성/업로드 단계 없음). `train_mm.py`가 `--seed_dataset`에서 이미지를 받아 학습합니다. gemma-4는 apache-2.0/ungated라 HF 토큰이 필요 없습니다.

In [ ]:
MAX_TRAIN_SAMPLES = 200   # 멀티모달은 무거우니 작게 시작. 정식은 None(전체).
MAX_RUNTIME_HOURS = 4   # 생략 시 SDK 기본 1시간 → 머지 중 강제 중단(docs/03 「MaxRuntimeExceeded」)
hyperparameters = {
    'model_id': config.DEFAULT_MODEL_ID,   # gemma-4 (멀티모달)
    'seed_dataset': TRACK.seed_dataset,
    'epochs': 2, 'per_device_train_batch_size': 1, 'gradient_accumulation_steps': 8,
    'learning_rate': 2e-4,
    'max_seq_length': 2048,
    'lora_r': 16, 'lora_alpha': 16, 'lora_dropout': 0.05,
    'use_qlora': True, 'freeze_vision': True, 'merge_adapter': True,
}
if MAX_TRAIN_SAMPLES:
    hyperparameters['max_train_samples'] = MAX_TRAIN_SAMPLES
environment = {'HF_TOKEN': config.get_hf_token()} if config.get_hf_token() else {}
image_uri = dlc.resolve_training_image(config.AWS_REGION)
assert image_uri, 'DLC 이미지 해석 실패: DLC_IMAGE_URI env로 지정: ' + dlc.AVAILABLE_IMAGES_URL
trainer = ModelTrainer(
    training_image=image_uri,
    source_code=SourceCode(source_dir='scripts', entry_script='train_mm.py',
                           requirements='requirements.txt'),   # torchvision 포함
    compute=Compute(instance_type=config.TRAIN_INSTANCE_TYPE, instance_count=1),
    hyperparameters=hyperparameters,
    environment=environment,
    role=role,
    sagemaker_session=sess,
    base_job_name='gemma-mm-extraction-train',
    stopping_condition=StoppingCondition(max_runtime_in_seconds=MAX_RUNTIME_HOURS * 3600),
)

## 학습 시작 (비동기): 데이터는 컨테이너가 직접 로드하므로 input_data 채널 불필요
이미지 시드를 컨테이너 안에서 `load_dataset`으로 받으므로 별도 train 채널을 붙이지 않습니다.

In [ ]:
trainer.train(wait=False, logs=False)
from IPython.display import display
job = trainer._latest_training_job
print('training job:', job.training_job_name)
display(aws_utils.cw_links(config.AWS_REGION, training_job=job.training_job_name))

### 진행 상태 확인 (이 셀만 반복 실행)
필요할 때마다 이 셀을 다시 실행해 진행 단계를 봅니다. `Starting → Pending(용량 대기) → Downloading(이미지 pull) → Training(코드 실행)` 순으로 진행되며, **Training 단계부터 CloudWatch 로그가 생깁니다**. 멀티모달 학습은 텍스트보다 오래 걸립니다.

In [ ]:
aws_utils.training_job_status(job.training_job_name, config.AWS_REGION)

### 세션이 끊겼을 때 잡에 다시 붙기 (재접속)
제출한 학습 잡은 SageMaker 서버에서 돌기 때문에 노트북 커널이 끊겨도 계속 진행됩니다. 다시 붙을 때 **train 셀을 재실행하면 GPU 잡이 중복 제출되어 비용이 두 배로 듭니다**: 대신 아래 셀로 잡 이름을 조회해 `job` 변수만 복구하세요. 위쪽 설정 셀만 실행한 상태에서 바로 쓸 수 있습니다.

In [ ]:
from sagemaker.core.resources import TrainingJob
# 방법 A: 잡 이름을 알면 바로 붙기 (가장 확실: 위 train 셀 출력에서 복사)
# job = TrainingJob.get('<여기에 잡 이름>')
# 방법 B: 이름을 잊었으면 base_job_name으로 최근 잡 찾기 (get_all은 최신순)
jobs = list(TrainingJob.get_all(name_contains='gemma-mm-extraction-train'))
assert jobs, '이 base_job_name으로 제출된 잡이 없습니다. 위 train 셀을 먼저 실행하세요.'
job = TrainingJob.get(jobs[0].get_name())
job.refresh()
print('reattached to:', job.training_job_name)
print('status       :', job.training_job_status, '/', job.secondary_status)
from IPython.display import display
display(aws_utils.cw_links(config.AWS_REGION, training_job=job.training_job_name))

## 학습 완료 대기 → 모델 아티팩트
잡이 끝나야 모델 아티팩트(S3)가 생깁니다. 아래 셀은 완료까지 폴링합니다. 지금 기다리지 않아도 되며, 나중에 위 재접속 셀로 `job`을 복구한 뒤 다시 실행하면 됩니다.

In [ ]:
import time
# `job`은 위 train 셀 또는 재접속 셀에서 정의됩니다(trainer 객체에 의존하지 않음).
assert 'job' in dir() and job is not None, (
    "job이 없습니다: 위 train 셀이나 '세션이 끊겼을 때 잡에 다시 붙기' 셀을 먼저 실행하세요.")
while True:
    job.refresh(); st = job.training_job_status; print('status:', st)
    if st in ('Completed','Failed','Stopped'): break
    time.sleep(30)
assert st == 'Completed', f'잡이 {st} 상태입니다. CloudWatch 로그 확인.'
model_data = job.model_artifacts.s3_model_artifacts
print('MM model artifact:', model_data)
md_mm_extraction = model_data   # 트랙 전용 키(전역은 다른 트랙이 덮어씀)
%store model_data
%store md_mm_extraction

멀티모달 학습이 끝났습니다. 산출물은 **멀티모달 그대로**(vision 포함) 저장됩니다. 다음은 **03_deploy_mm_endpoint.ipynb**로 이미지 입력을 받는 endpoint를 배포합니다.